In [ ]:
rm -rf /content/*


In [ ]:
pip install -U ultralytics pycocotools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00


In [ ]:
%%bash
mkdir -p /content/coco
cd /content/coco

wget http://images.cocodataset.org/zips/val2017.zip
wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

unzip val2017.zip
unzip annotations_trainval2017.zip


Archive:  val2017.zip
   creating: val2017/
 extracting: val2017/000000212226.jpg  
 extracting: val2017/000000231527.jpg  
 extracting: val2017/000000578922.jpg  
 extracting: val2017/000000062808.jpg  
 extracting: val2017/000000119038.jpg  
 extracting: val2017/000000114871.jpg  
 extracting: val2017/000000463918.jpg  
 extracting: val2017/000000365745.jpg  
 extracting: val2017/000000320425.jpg  
 extracting: val2017/000000481404.jpg  
 extracting: val2017/000000314294.jpg  
 extracting: val2017/000000335328.jpg  
 extracting: val2017/000000513688.jpg  
 extracting: val2017/000000158548.jpg  
 extracting: val2017/000000132116.jpg  
 extracting: val2017/000000415238.jpg  
 extracting: val2017/000000321333.jpg  
 extracting: val2017/000000081738.jpg  
 extracting: val2017/000000577584.jpg  
 extracting: val2017/000000346905.jpg  
 extracting: val2017/000000433980.jpg  
 extracting: val2017/000000228144.jpg  
 extracting: val2017/000000041872.jpg  
 extracting: val2017/000000117492.jp

--2026-01-26 07:28:55--  http://images.cocodataset.org/zips/val2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.182.41.241, 16.182.108.49, 16.182.74.65, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.182.41.241|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 815585330 (778M) [application/zip]
Saving to: ‘val2017.zip’

     0K .......... .......... .......... .......... ..........  0%  412K 32m15s
    50K .......... .......... .......... .......... ..........  0%  824K 24m10s
   100K .......... .......... .......... .......... ..........  0% 67.2M 16m11s
   150K .......... .......... .......... .......... ..........  0% 59.5M 12m11s
   200K .......... .......... .......... .......... ..........  0%  841K 12m54s
   250K .......... .......... .......... .......... ..........  0% 97.8M 10m46s
   300K .......... .......... .......... .......... ..........  0%  205M 9m15s
   350K .......... .......... .......... .........

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate_coco(result_json):
    coco_gt = COCO("/content/coco/annotations/instances_val2017.json")
    coco_dt = coco_gt.loadRes(result_json)

    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval.stats[0]  # mAP


In [ ]:
import torch, json, time
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
class CocoDataset(CocoDetection):
    def __getitem__(self, idx):
        img, target = super().__getitem__(idx)
        img = F.to_tensor(img)
        image_id = self.ids[idx]
        return img, image_id

dataset = CocoDataset(
    root="/content/coco/val2017",
    annFile="/content/coco/annotations/instances_val2017.json"
)

# 20%만 사용
subset_size = int(len(dataset) * 0.2)
dataset = torch.utils.data.Subset(dataset, range(subset_size))

loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Model
model = fasterrcnn_resnet50_fpn(weights="DEFAULT").to(device).eval()

results = []
start = time.time()

with torch.no_grad():
    for images, image_ids in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for output, image_id in zip(outputs, image_ids):
            for box, score, label in zip(output["boxes"], output["scores"], output["labels"]):
                if score < 0.05:
                    continue
                x1, y1, x2, y2 = box.tolist()
                results.append({
                    "image_id": int(image_id),
                    "category_id": int(label),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": float(score)
                })

frcnn_time = time.time() - start

with open("frcnn_results.json", "w") as f:
    json.dump(results, f)

frcnn_map = evaluate_coco("frcnn_results.json")

print(f"Faster R-CNN | mAP: {frcnn_map:.3f} | time: {frcnn_time:.2f}s")


loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 199MB/s]


loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.47s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=9.99s).
Accumulating evaluation results...
DONE (t=1.57s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.079
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.121
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.086
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.041
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.087
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.104
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.063
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.098
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [ ]:
%%writefile /content/coco_yolo.yaml
path: /content/coco
train: images/train2017
val: images/val2017

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush


Writing /content/coco_yolo.yaml


In [ ]:
from ultralytics import YOLO
import time

model = YOLO("yolov8n.pt")

start = time.time()
metrics = model.val(
    data="/content/coco_yolo.yaml",
    split="val",
    fraction=0.2,
    save_json=True,
    save=False,
    plots=False
)
yolo_time = time.time() - start

yolo_map = metrics.box.map  # ⭐ YOLO가 직접 계산한 mAP

print(f"YOLOv8 | mAP: {yolo_map:.3f} | time: {yolo_time:.2f}s")


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 51.8±15.7 MB/s, size: 172.5 KB)
val: Scanning /content/coco/labels/val2017... 0 images, 5000 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 496.7it/s 10.1s
WARNING ⚠️ val: No labels found in /content/coco/labels/val2017.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
WARNING ⚠️ val: Cache directory /content/coco/labels is not writable, cache not saved.
WARNING ⚠️ Labels are missing or empty in /content/coco/labels/val2017.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 8.1it/s 38.9s
                   all       5000          0          0          0          0

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:837: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


Results saved to /content/runs/detect/val2
YOLOv8 | mAP: 0.000 | time: 58.84s


In [ ]:
from ultralytics import YOLO
import time

model = YOLO("rtdetr-l.pt")

start = time.time()
metrics = model.val(
    data="/content/coco_yolo.yaml",   # 🔥 Ultralytics 공식 COCO
    split="val",
    fraction=0.2,       # 일부만 (속도용)
    save=False,
    plots=False
)
rtdetr_time = time.time() - start

rtdetr_map = metrics.box.map  # ⭐ 이게 정답

print(f"RT-DETR | mAP: {rtdetr_map:.3f} | time: {rtdetr_time:.2f}s")


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
rt-detr-l summary: 294 layers, 32,148,140 parameters, 0 gradients, 103.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1910.1±313.6 MB/s, size: 107.0 KB)
val: Scanning /content/coco/labels/val2017... 0 images, 5000 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 5.1Kit/s 1.0s
WARNING ⚠️ val: No labels found in /content/coco/labels/val2017.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
WARNING ⚠️ val: Cache directory /content/coco/labels is not writable, cache not saved.
WARNING ⚠️ Labels are missing or empty in /content/coco/labels/val2017.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 1.6it/s 3:17
                   all       5000          0          0          0          0  

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:837: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [ ]:
print("\n📊 Detection Model Comparison")
print("----------------------------------------")
print(f"Faster R-CNN | mAP: {frcnn_map:.3f} | time: {frcnn_time:.2f}s")
print(f"YOLOv8      | mAP: {yolo_map:.3f} | time: {yolo_time:.2f}s")
print(f"RT-DETR     | mAP: {rtdetr_map:.3f} | time: {rtdetr_time:.2f}s")



📊 Detection Model Comparison
----------------------------------------
Faster R-CNN | mAP: 0.079 | time: 141.04s
YOLOv8      | mAP: 0.000 | time: 58.84s
RT-DETR     | mAP: 0.000 | time: 202.52s


In [ ]:
from ultralytics import YOLO
import time

model = YOLO("yolov8n.pt")

start = time.time()
metrics = model.val(
    data="coco_yolo.yaml",   # ← 그대로 사용
    split="val",
    save_json=True,
    fraction=0.2,
    project="runs/compare",
    name="yolo"
)
elapsed = time.time() - start

print(f"YOLOv8 | mAP50-95: {metrics.box.map:.3f} | time: {elapsed:.2f}s")


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2316.6±296.0 MB/s, size: 143.1 KB)
val: Scanning /content/coco/labels/val2017... 0 images, 5000 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 5.1Kit/s 1.0s
WARNING ⚠️ val: No labels found in /content/coco/labels/val2017.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
WARNING ⚠️ val: Cache directory /content/coco/labels is not writable, cache not saved.
WARNING ⚠️ Labels are missing or empty in /content/coco/labels/val2017.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 7.6it/s 41.2s


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:655: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:701: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:701: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultraly

                   all       5000          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 0.8ms preprocess, 2.8ms inference, 0.0ms loss, 1.2ms postprocess per image
Saving /content/runs/detect/runs/compare/yolo/predictions.json...
Results saved to /content/runs/detect/runs/compare/yolo
YOLOv8 | mAP50-95: 0.000 | time: 54.77s


In [ ]:
from ultralytics import YOLO
import time

model = YOLO("rtdetr-l.pt")

start = time.time()
metrics_rtdetr = model.val(
    data="coco_yolo.yaml",
    split="val",
    save_json=True,
    fraction=0.2,
    project="runs/compare",
    name="rtdetr"
)
rtdetr_time = time.time() - start

rtdetr_map = metrics_rtdetr.box.map

print(f"RT-DETR | mAP: {rtdetr_map:.3f} | time: {rtdetr_time:.2f}s")


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
rt-detr-l summary: 294 layers, 32,148,140 parameters, 0 gradients, 103.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2766.5±787.6 MB/s, size: 202.3 KB)
val: Scanning /content/coco/labels/val2017... 0 images, 5000 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000 5.2Kit/s 1.0s
WARNING ⚠️ val: No labels found in /content/coco/labels/val2017.cache. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
WARNING ⚠️ val: Cache directory /content/coco/labels is not writable, cache not saved.
WARNING ⚠️ Labels are missing or empty in /content/coco/labels/val2017.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 1.5it/s 3:28


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:655: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:701: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:701: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultraly

                   all       5000          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 0.9ms preprocess, 37.4ms inference, 0.0ms loss, 0.4ms postprocess per image
Saving /content/runs/detect/runs/compare/rtdetr/predictions.json...
Results saved to /content/runs/detect/runs/compare/rtdetr
RT-DETR | mAP: 0.000 | time: 230.32s


In [ ]:
!find /content -name "instances_val2017.json"


/content/coco/annotations/instances_val2017.json


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def coco_map(pred_json_path):
    gt = COCO("/content/coco/annotations/instances_val2017.json")
    dt = gt.loadRes(pred_json_path)

    coco_eval = COCOeval(gt, dt, "bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval.stats[0]  # mAP@[0.5:0.95]


In [ ]:
!find runs -name "predictions.json"


runs/detect/val10/predictions.json
runs/detect/val2/predictions.json
runs/detect/runs/compare/rtdetr/predictions.json
runs/detect/runs/compare/yolo/predictions.json
runs/detect/val11/predictions.json
runs/detect/val13/predictions.json


In [ ]:
yolo_map = coco_map("runs/detect/runs/compare/yolo/predictions.json")
rtdetr_map = coco_map("runs/detect/runs/compare/rtdetr/predictions.json")

print(f"YOLOv8   mAP: {yolo_map:.3f}")
print(f"RT-DETR  mAP: {rtdetr_map:.3f}")


loading annotations into memory...
Done (t=0.39s)
creating index...
index created!
Loading and preparing results...
DONE (t=3.25s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=39.62s).
Accumulating evaluation results...
DONE (t=11.79s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.062
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.088
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.066
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.031
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.062
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.092
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.072
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.125
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

In [ ]:
print("\n📊 Detection Model Comparison")
print("----------------------------------------")
print(f"Faster R-CNN | mAP: {frcnn_map:.3f} | time: {frcnn_time:.2f}s")
print(f"YOLOv8      | mAP: {yolo_map:.3f} | time: {yolo_time:.2f}s")
print(f"RT-DETR     | mAP: {rtdetr_map:.3f} | time: {rtdetr_time:.2f}s")



📊 Detection Model Comparison
----------------------------------------
Faster R-CNN | mAP: 0.079 | time: 141.04s
YOLOv8      | mAP: 0.062 | time: 58.84s
RT-DETR     | mAP: 0.078 | time: 230.32s
